In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import networkx as nx
import sys
import pyscf
import scipy
import ffsim

from itertools import combinations

sys.path.insert(0, "..")

In [2]:
import optimize

In [3]:
x = np.loadtxt("../outputs/20260602_121502_x_opt.txt", skiprows=1)

In [5]:
x.shape

(47,)

In [7]:
len(list(combinations(range(10), 2)))

45

In [9]:
U = optimize.x_to_rotation(x[:-2], 10)

In [13]:
scf = pyscf.tools.fcidump.to_scf("../hamiltonians/fe2s2_10e10o.FCIDUMP")
moldata = ffsim.MolecularData.from_scf(scf)

Parsing ../hamiltonians/fe2s2_10e10o.FCIDUMP
converged SCF energy = -116.154925541542


Overwritten attributes  get_ovlp get_hcore  of <class 'pyscf.scf.hf_symm.SymAdaptedRHF'>
/home/alexey/quantchem/lib/python3.12/site-packages/pyscf/gto/mole.py:1300: UserWarning: Function mol.dumps drops attribute energy_nuc because it is not JSON-serializable
  warnings.warn(msg)
/home/alexey/quantchem/lib/python3.12/site-packages/pyscf/gto/mole.py:1300: UserWarning: Function mol.dumps drops attribute intor_symmetric because it is not JSON-serializable
  warnings.warn(msg)


In [14]:
h = ffsim.linear_operator(moldata.hamiltonian,
                                      norb=moldata.norb, nelec=moldata.nelec)

In [15]:
cisolver = pyscf.fci.FCI(scf)
cisolver.kernel()

(-116.51635833557486,
 FCIvector([[ 6.63232587e-05, -1.74722033e-07,  5.05362549e-07, ...,
              1.38810252e-03, -7.45758696e-02,  2.34788818e-04],
            [-1.84787657e-07, -2.46190531e-05, -1.02645438e-07, ...,
             -7.18661013e-05,  2.19668267e-04,  4.30239833e-02],
            [ 5.72737121e-07, -1.96464326e-07, -2.86352555e-05, ...,
             -4.40818937e-03,  5.46013274e-05,  5.37355161e-04],
            ...,
            [ 1.37008556e-03, -7.23241958e-05, -4.36176409e-03, ...,
              2.81285642e-05, -3.17842611e-07, -2.50465467e-07],
            [-7.40024296e-02,  2.21817485e-04,  5.49592126e-05, ...,
              7.03462885e-07,  7.14520224e-06, -3.28579652e-08],
            [ 2.28692896e-04,  4.26693370e-02,  5.35857076e-04, ...,
             -5.18494065e-08,  3.85707150e-08, -3.40453702e-06]]))

In [16]:
state = np.array(cisolver.ci.reshape((-1,)), dtype="complex")

In [17]:
state.shape

(63504,)

In [18]:
state.conj() @ h @ state

(-116.51635833557484+0j)

In [19]:
rotated_h = ffsim.linear_operator(moldata.hamiltonian.rotated(U),
                                      norb=moldata.norb, nelec=moldata.nelec)

In [20]:
rotated_state = ffsim.apply_orbital_rotation(state, U, moldata.norb, moldata.nelec)

In [21]:
rotated_state.conj() @ rotated_h @ rotated_state

(-116.51635833557488+0j)

In [22]:
rotated_hamiltonian = moldata.hamiltonian.rotated(U)

In [30]:
dir(rotated_hamiltonian)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__class_getitem__',
 '__dataclass_fields__',
 '__dataclass_params__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__match_args__',
 '__module__',
 '__ne__',
 '__new__',
 '__parameters__',
 '__protocol_attrs__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__slots__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_approx_eq_',
 '_diag_',
 '_fermion_operator_',
 '_is_protocol',
 '_is_runtime_protocol',
 '_linear_operator_',
 '_trace_',
 'constant',
 'from_fermion_operator',
 'norb',
 'one_body_tensor',
 'one_body_tensor_spinless',
 'rotated',
 'two_body_tensor',
 'two_body_tensor_spinless']

In [32]:
rot_moldata = ffsim.MolecularData(
    atom=moldata.atom,
    basis=moldata.basis,
    spin=moldata.spin,
    nelec=moldata.nelec,
    hf_energy=moldata.hf_energy,
    one_body_integrals=rotated_hamiltonian.one_body_tensor,
    two_body_integrals=rotated_hamiltonian.two_body_tensor,
    norb = moldata.norb,
    core_energy=moldata.core_energy
)

In [35]:
h2 = ffsim.linear_operator(rot_moldata.hamiltonian,
                                      norb=moldata.norb, nelec=moldata.nelec)

In [36]:
rotated_state.conj() @ h2 @ rotated_state

(-116.51635833557488+0j)

In [37]:
reverse_rotated_state = ffsim.apply_orbital_rotation(state, U.T.conj(), moldata.norb, moldata.nelec)

In [38]:
reverse_rotated_state.conj() @ h2 @ reverse_rotated_state

(-116.08832635724548+0j)

In [41]:
rot_moldata.to_fcidump("fe2s2_10o10e_CASSCF_rotated_to_C_cost.FCIDUMP")